# Multi-Environment RLHF Notebook (Hopper + Pendulum)

This notebook is organized as:
1. Shared definitions only (no training runs)
2. Environment-specific configs and prompt styles
3. Run cells for different experiment types

In [ ]:
# Install required packages
!pip install -q "gymnasium[mujoco]"
!pip install -q -U transformers accelerate

In [ ]:
# Shared imports and global hyperparameters (definitions only)
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
import gymnasium as gym

# Model/training hyperparameters
HIDDEN_DIM = 128
LR_POLICY = 3e-4
LR_VALUE = 1e-3
LR_REWARD = 1e-3
GAMMA = 0.99
LAM = 0.95
PPO_EPOCHS = 4
CLIP_EPS = 0.2
VALUE_COEF = 0.5
ENTROPY_COEF = 0.01
MAX_GRAD_NORM = 0.5
SEGMENT_LENGTH = 50
ROLLOUT_STEPS = 2048
COMPARISONS_PER_ROUND = 20
RM_TRAIN_EPOCHS = 10

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def parse_preference_from_llm(raw: str):
    """Parse A/B preference from model output."""
    match = re.match(r"^\s*([AB])\b", raw, re.IGNORECASE)
    if match:
        preference = match.group(1).upper()
        explanation = raw[match.end():].strip().lstrip(".,:-")
        return preference, explanation

    letters = re.findall(r"\b([AB])\b", raw, re.IGNORECASE)
    preference = letters[0].upper() if letters else "UNCLEAR"
    return preference, raw

In [ ]:
# Environment-specific configuration and prompt templates (definitions only)
ENV_CONFIGS = {
    "hopper": {
        "env_id": "Hopper-v5",
        "obs_dim": 11,
        "action_dim": 3,
        "forward_index": 5,
        "height_index": 0,
        "angle_index": 1,
        "healthy_height_min": 0.7,
        "healthy_angle_min": -0.2,
        "healthy_angle_max": 0.2,
        "terminated_can_mean_fall": True,
    },
    "pendulum": {
        "env_id": "Pendulum-v1",
        "obs_dim": 3,
        "action_dim": 1,
        "cos_index": 0,
        "sin_index": 1,
        "theta_dot_index": 2,
        "terminated_can_mean_fall": False,
        "upright_cos_threshold": 0.9,
        "still_theta_dot_threshold": 1.0,
    },
    "ant":{
        "env_id": "Ant-v5",
        "obs_dim": 27,
        "action_dim": 8,
        "x_velocity_index": 13,
        "z_height_index": 0,
        "healthy_height_min": 0.2,
        "healthy_height_max": 1.0,
        "terminated_can_mean_fall": True,
    }
}

PROMPT_STYLES_BY_ENV = {
    "hopper": {
        "balanced_goal_first": (
            "You are evaluating two Hopper trajectory segments.\n"
            "Goal: stay upright and balanced while moving forward.\n"
            "Healthy posture: torso height > {healthy_height_min:.2f} and torso angle in [{healthy_angle_min:.2f}, {healthy_angle_max:.2f}].\n\n"
            "{desc_a}\n\n{desc_b}\n\n"
            "Which segment is better overall?"
        ),
        "checklist_style": (
            "Evaluate Segment A and Segment B for Hopper.\n"
            "Checklist in order: (1) upright stability, (2) tilt control, (3) forward movement consistency.\n\n"
            "{desc_a}\n\n{desc_b}\n\n"
            "Which segment should be preferred?"
        ),
        "minimal_stats_style": (
            "Choose the better Hopper segment for robust locomotion under noise.\n\n"
            "{desc_a}\n\n{desc_b}\n\n"
            "Choose one."
        ),
    },
    "pendulum": {
        "balanced_goal_first": (
            "You are evaluating two Pendulum control segments.\n"
            "Goal: keep the pendulum upright with low angular velocity and smooth torque.\n\n"
            "{desc_a}\n\n{desc_b}\n\n"
            "Which segment better achieves stable upright control?"
        ),
        "checklist_style": (
            "Evaluate Segment A and Segment B for Pendulum.\n"
            "Checklist: (1) uprightness, (2) low oscillation, (3) lower control effort.\n\n"
            "{desc_a}\n\n{desc_b}\n\n"
            "Which segment is better?"
        ),
        "minimal_stats_style": (
            "Pick the segment with more stable upright behavior and less swinging.\n\n"
            "{desc_a}\n\n{desc_b}\n\n"
            "Choose one."
        ),
    },
    "ant":{
        "balanced_goal_first": (
            "You are evaluating two Ant robot trajectory segments.\n"
            "Goal: move forward efficiently while maintaining stability (not falling).\n"
            "Healthy posture: torso height within a reasonable range.\n\n"
            "{desc_a}\n\n{desc_b}\n\n"
            "Which segment is better overall?"
        ),
        "checklist_style": (
            "Evaluate Segment A and Segment B for Ant.\n"
            "Checklist: (1) stability (no falling), (2) forward movement, (3) smooth motion.\n\n"
            "{desc_a}\n\n{desc_b}\n\n"
            "Which segment is better?"
        ),
        "minimal_stats_style": (
            "Pick the better Ant segment for stable forward locomotion.\n\n"
            "{desc_a}\n\n{desc_b}\n\n"
            "Choose one."
        ),
    }
}

OUTPUT_INSTRUCTION = "\n\nReturn ONLY: the letter A or B, then one short sentence explanation."

def make_env(env_name: str, seed: int = None):
    cfg = ENV_CONFIGS[env_name]
    env = gym.make(cfg["env_id"])
    if seed is not None:
        env.reset(seed=seed)
    return env, cfg

In [ ]:
# Core models and preference buffer (definitions only)
class PolicyNetwork(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, HIDDEN_DIM),
            nn.Tanh(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.Tanh(),
        )
        self.mean_head = nn.Linear(HIDDEN_DIM, action_dim)
        self.log_std_head = nn.Linear(HIDDEN_DIM, action_dim)
        nn.init.orthogonal_(self.mean_head.weight, gain=0.01)
        nn.init.zeros_(self.mean_head.bias)

    def forward(self, obs):
        x = self.net(obs)
        mean = self.mean_head(x)
        log_std = self.log_std_head(x)
        log_std = torch.clamp(log_std, min=-2.0, max=0.5)
        std = torch.exp(log_std)
        return Normal(mean, std)

    def get_action(self, obs):
        obs_t = torch.FloatTensor(obs).unsqueeze(0)
        dist = self.forward(obs_t)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        action_np = action.squeeze(0).detach().numpy()
        action_np = np.clip(action_np, -1.0, 1.0)
        return action_np, log_prob, entropy

    def evaluate_actions(self, obs, actions):
        dist = self.forward(obs)
        log_probs = dist.log_prob(actions).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        return log_probs, entropy


class ValueNetwork(nn.Module):
    def __init__(self, obs_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, HIDDEN_DIM),
            nn.Tanh(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.Tanh(),
            nn.Linear(HIDDEN_DIM, 1),
        )
        nn.init.orthogonal_(self.net[-1].weight, gain=1.0)

    def forward(self, obs):
        return self.net(obs)


class RewardModel(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        input_dim = obs_dim + action_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, HIDDEN_DIM),
            nn.Tanh(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.Tanh(),
            nn.Linear(HIDDEN_DIM, 1),
        )
        nn.init.orthogonal_(self.net[-1].weight, gain=1.0)

    def score_segment(self, obs_seq, act_seq):
        x = torch.cat([obs_seq, act_seq], dim=-1)
        return self.net(x).sum()

    def forward(self, obs_seq, act_seq):
        return self.score_segment(obs_seq, act_seq)


class ComparisonBuffer:
    def __init__(self):
        self.winners = []
        self.losers = []

    def __len__(self):
        return len(self.winners)

    def add(self, seg_a, seg_b, preference):
        if preference == "A":
            winner, loser = seg_a, seg_b
        elif preference == "B":
            winner, loser = seg_b, seg_a
        else:
            return

        self.winners.append((torch.FloatTensor(winner["observations"]), torch.FloatTensor(winner["actions"])))
        self.losers.append((torch.FloatTensor(loser["observations"]), torch.FloatTensor(loser["actions"])))

    def train_reward_model(self, reward_model, num_epochs=5, lr=1e-3):
        if len(self) == 0:
            return []

        optimizer = torch.optim.Adam(reward_model.parameters(), lr=lr)
        losses = []
        for _ in range(num_epochs):
            epoch_loss = 0.0
            for (win_obs, win_act), (lose_obs, lose_act) in zip(self.winners, self.losers):
                r_winner = reward_model.score_segment(win_obs, win_act)
                r_loser = reward_model.score_segment(lose_obs, lose_act)
                loss = -F.logsigmoid(r_winner - r_loser)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            losses.append(epoch_loss / len(self))

        return losses

In [ ]:
# Segment stats, descriptions, prompts, rollout collection (definitions only)
def _pendulum_angle_from_obs(obs_arr, cfg):
    cos_th = obs_arr[:, cfg["cos_index"]]
    sin_th = obs_arr[:, cfg["sin_index"]]
    return np.arctan2(sin_th, cos_th)


def compute_segment_stats_env(obs_arr, act_arr, env_rewards, terminated_flag, cfg):
    seg = {
        "observations": obs_arr,
        "actions": act_arr,
        "segment_length": int(len(obs_arr)),
        "env_avg_reward": float(np.mean(env_rewards)) if len(env_rewards) else 0.0,
        "env_total_reward": float(np.sum(env_rewards)) if len(env_rewards) else 0.0,
    }

    if cfg["env_id"] == "Hopper-v5":
        h_idx = cfg["height_index"]
        a_idx = cfg["angle_index"]
        f_idx = cfg["forward_index"]
        healthy_height = obs_arr[:, h_idx] > cfg["healthy_height_min"]
        healthy_angle = (obs_arr[:, a_idx] >= cfg["healthy_angle_min"]) & (obs_arr[:, a_idx] <= cfg["healthy_angle_max"])
        seg.update({
            "fell": bool(terminated_flag),
            "time_upright": int(np.sum(healthy_height)),
            "avg_height": float(np.mean(obs_arr[:, h_idx])),
            "min_height": float(np.min(obs_arr[:, h_idx])),
            "max_height": float(np.max(obs_arr[:, h_idx])),
            "avg_angle": float(np.mean(obs_arr[:, a_idx])),
            "min_angle": float(np.min(obs_arr[:, a_idx])),
            "max_angle": float(np.max(obs_arr[:, a_idx])),
            "angle_healthy_pct": float(np.mean(healthy_angle) * 100.0),
            "avg_forward_metric": float(np.mean(obs_arr[:, f_idx])),
            "forward_metric_name": "avg_forward_velocity",
        })
    elif cfg["env_id"] == "Pendulum-v1":
        theta = _pendulum_angle_from_obs(obs_arr, cfg)
        theta_dot = obs_arr[:, cfg["theta_dot_index"]]
        upright = obs_arr[:, cfg["cos_index"]] >= cfg["upright_cos_threshold"]
        still = np.abs(theta_dot) <= cfg["still_theta_dot_threshold"]
        seg.update({
            "fell": False,
            "time_upright": int(np.sum(upright)),
            "avg_abs_angle": float(np.mean(np.abs(theta))),
            "max_abs_angle": float(np.max(np.abs(theta))),
            "avg_abs_theta_dot": float(np.mean(np.abs(theta_dot))),
            "still_pct": float(np.mean(still) * 100.0),
            "avg_forward_metric": float(-np.mean(np.abs(theta))),
            "forward_metric_name": "negative_abs_angle",
        })
    elif cfg["env_id"] == "Ant-v5":
        h_idx = cfg["z_height_index"]
        v_idx = cfg["x_velocity_index"]

        healthy_height = (
            (obs_arr[:, h_idx] > cfg["healthy_height_min"]) &
            (obs_arr[:, h_idx] < cfg["healthy_height_max"])
        )

        seg.update({
            "fell": bool(terminated_flag),
            "time_healthy": int(np.sum(healthy_height)),
            "avg_height": float(np.mean(obs_arr[:, h_idx])),
            "min_height": float(np.min(obs_arr[:, h_idx])),
            "max_height": float(np.max(obs_arr[:, h_idx])),
            "avg_forward_metric": float(np.mean(obs_arr[:, v_idx])),
            "forward_metric_name": "x_velocity",
        })
    else:
        raise ValueError(f"Unsupported env id: {cfg['env_id']}")

    return seg


def describe_segment_env(seg, cfg, label="A"):
    if cfg["env_id"] == "Hopper-v5":
        return (
            f"Segment {label} ({seg['segment_length']} steps):\n"
            f"- Fell: {seg['fell']}\n"
            f"- Upright steps: {seg['time_upright']}/{seg['segment_length']}\n"
            f"- Height avg/min/max: {seg['avg_height']:.3f}/{seg['min_height']:.3f}/{seg['max_height']:.3f}\n"
            f"- Angle avg/min/max: {seg['avg_angle']:.3f}/{seg['min_angle']:.3f}/{seg['max_angle']:.3f}\n"
            f"- Angle healthy %: {seg['angle_healthy_pct']:.1f}%\n"
            f"- Avg forward velocity: {seg['avg_forward_metric']:.3f}\n"
            f"- Env total reward: {seg['env_total_reward']:.3f}"
        )

    if cfg["env_id"] == "Pendulum-v1":
        return (
            f"Segment {label} ({seg['segment_length']} steps):\n"
            f"- Upright steps: {seg['time_upright']}/{seg['segment_length']}\n"
            f"- Avg |angle| (rad): {seg['avg_abs_angle']:.3f}\n"
            f"- Max |angle| (rad): {seg['max_abs_angle']:.3f}\n"
            f"- Avg |theta_dot|: {seg['avg_abs_theta_dot']:.3f}\n"
            f"- Stillness %: {seg['still_pct']:.1f}%\n"
            f"- Env total reward: {seg['env_total_reward']:.3f}"
        )
    if cfg["env_id"] == "Ant-v5":
        return (
            f"Segment {label} ({seg['segment_length']} steps):\n"
            f"- Fell: {seg['fell']}\n"
            f"- Healthy steps: {seg['time_healthy']}/{seg['segment_length']}\n"
            f"- Height avg/min/max: {seg['avg_height']:.3f}/{seg['min_height']:.3f}/{seg['max_height']:.3f}\n"
            f"- Avg forward velocity: {seg['avg_forward_metric']:.3f}\n"
            f"- Env total reward: {seg['env_total_reward']:.3f}"
        )
    raise ValueError(f"Unsupported env id: {cfg['env_id']}")


def build_pair_prompt(seg_a, seg_b, env_name, style_name):
    cfg = ENV_CONFIGS[env_name]
    styles = PROMPT_STYLES_BY_ENV[env_name]
    if style_name not in styles:
        raise ValueError(f"Unknown style '{style_name}' for env '{env_name}'")

    desc_a = describe_segment_env(seg_a, cfg, label="A")
    desc_b = describe_segment_env(seg_b, cfg, label="B")
    prompt = styles[style_name].format(desc_a=desc_a, desc_b=desc_b, **cfg)
    return prompt + OUTPUT_INSTRUCTION


def get_llm_preference_with_style(seg_a, seg_b, env_name, style_name, llm_query_fn, max_new_tokens=80):
    prompt = build_pair_prompt(seg_a, seg_b, env_name=env_name, style_name=style_name)
    raw = llm_query_fn(prompt, max_new_tokens=max_new_tokens)
    preference, explanation = parse_preference_from_llm(raw)
    return preference, explanation, raw


def collect_rollout(env, cfg, policy, value_net, reward_model=None, rollout_steps=ROLLOUT_STEPS, segment_length=SEGMENT_LENGTH):
    obs_list, act_list, logprob_list = [], [], []
    rm_reward_list, env_reward_list, value_list, done_list = [], [], [], []

    segments = []
    seg_obs, seg_act, seg_env_rewards = [], [], []

    obs, _ = env.reset()

    for _ in range(rollout_steps):
        obs_t = torch.FloatTensor(obs)
        with torch.no_grad():
            action_np, log_prob, _ = policy.get_action(obs)
            value = value_net(obs_t.unsqueeze(0)).squeeze()

        next_obs, env_reward, terminated, truncated, _ = env.step(action_np)
        done = bool(terminated or truncated)

        if reward_model is not None:
            with torch.no_grad():
                rm_reward = reward_model(
                    torch.FloatTensor(obs).unsqueeze(0),
                    torch.FloatTensor(action_np).unsqueeze(0),
                ).item()
        else:
            rm_reward = float(env_reward)

        obs_list.append(obs.copy())
        act_list.append(action_np.copy())
        logprob_list.append(log_prob.item())
        rm_reward_list.append(float(rm_reward))
        env_reward_list.append(float(env_reward))
        value_list.append(value.item())
        done_list.append(float(done))

        seg_obs.append(obs.copy())
        seg_act.append(action_np.copy())
        seg_env_rewards.append(float(env_reward))

        if len(seg_obs) == segment_length:
            segments.append(
                compute_segment_stats_env(
                    np.array(seg_obs),
                    np.array(seg_act),
                    np.array(seg_env_rewards),
                    terminated_flag=terminated,
                    cfg=cfg,
                )
            )
            seg_obs, seg_act, seg_env_rewards = [], [], []

        obs = next_obs
        if done:
            obs, _ = env.reset()

    return {
        "observations": np.array(obs_list),
        "actions": np.array(act_list),
        "logprobs": np.array(logprob_list),
        "rewards": np.array(rm_reward_list),
        "env_rewards": np.array(env_reward_list),
        "values": np.array(value_list),
        "dones": np.array(done_list, dtype=np.float32),
        "segments": segments,
    }

In [ ]:
# PPO, training loops, and comparisons (definitions only)
def compute_gae(rewards, values, dones, gamma=GAMMA, lam=LAM):
    advantages = []
    gae = 0.0
    for t in reversed(range(len(rewards))):
        next_value = 0.0 if t == len(rewards) - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * gae * (1 - dones[t])
        advantages.insert(0, gae)

    advantages = torch.FloatTensor(advantages)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    returns = advantages + torch.FloatTensor(values)
    return advantages, returns


def ppo_update(policy, value_net, obs_batch, act_batch, old_logprobs, advantages, returns, ppo_epochs=PPO_EPOCHS, clip_eps=CLIP_EPS):
    policy_optimizer = torch.optim.Adam(policy.parameters(), lr=LR_POLICY)
    value_optimizer = torch.optim.Adam(value_net.parameters(), lr=LR_VALUE)

    obs_t = torch.FloatTensor(obs_batch)
    act_t = torch.FloatTensor(act_batch)
    old_lp_t = torch.FloatTensor(old_logprobs)

    for _ in range(ppo_epochs):
        new_log_probs, entropy = policy.evaluate_actions(obs_t, act_t)
        ratio = torch.exp(new_log_probs - old_lp_t)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages

        policy_loss = -torch.min(surr1, surr2).mean()
        entropy_loss = -ENTROPY_COEF * entropy.mean()
        policy_objective = policy_loss + entropy_loss

        policy_optimizer.zero_grad()
        policy_objective.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), MAX_GRAD_NORM)
        policy_optimizer.step()

        value_pred = value_net(obs_t).squeeze()
        value_loss = F.mse_loss(value_pred, returns)

        value_optimizer.zero_grad()
        value_loss.backward()
        torch.nn.utils.clip_grad_norm_(value_net.parameters(), MAX_GRAD_NORM)
        value_optimizer.step()


# Add-on evaluation: episode survival duration (in addition to current rollout metrics)
# Only environments listed here will collect survival metrics.
SURVIVAL_METRIC_ENVS = {'hopper', 'ant'}


def evaluate_episode_length_until_done(env, policy, num_episodes=10, max_steps=None):
    """Measure per-episode survival length until terminated/truncated (or optional max_steps cap)."""
    episode_lengths = []

    for _ in range(num_episodes):
        obs, _ = env.reset()
        steps = 0

        while True:
            with torch.no_grad():
                action_np, _, _ = policy.get_action(obs)

            obs, _, terminated, truncated, _ = env.step(action_np)
            steps += 1

            reached_user_cap = (max_steps is not None and steps >= max_steps)
            if terminated or truncated or reached_user_cap:
                break

        episode_lengths.append(steps)

    if not episode_lengths:
        return {
            'episode_length_mean': 0.0,
            'episode_length_median': 0.0,
            'episode_length_max': 0,
            'episode_length_min': 0,
            'episode_length_std': 0.0,
            'survival_eval_episodes': 0,
        }

    arr = np.array(episode_lengths, dtype=np.float32)
    return {
        'episode_length_mean': float(np.mean(arr)),
        'episode_length_median': float(np.median(arr)),
        'episode_length_max': int(np.max(arr)),
        'episode_length_min': int(np.min(arr)),
        'episode_length_std': float(np.std(arr)),
        'survival_eval_episodes': int(len(episode_lengths)),
    }


def add_survival_metrics_to_row(row, env_name, env, policy, survival_eval_episodes=10, survival_max_steps=None):
    """Attach survival keys to every round row for consistent data_points schema."""
    row['survival_metrics_collected'] = False
    row['episode_length_mean'] = None
    row['episode_length_median'] = None
    row['episode_length_max'] = None
    row['episode_length_min'] = None
    row['episode_length_std'] = None
    row['survival_eval_episodes'] = 0

    if env_name in SURVIVAL_METRIC_ENVS:
        row.update(
            evaluate_episode_length_until_done(
                env,
                policy,
                num_episodes=survival_eval_episodes,
                max_steps=survival_max_steps,
            )
        )
        row['survival_metrics_collected'] = True



def training_loop_no_llm(
    env, cfg, policy, value_net, num_rounds=5, rollout_steps=ROLLOUT_STEPS,
    env_name='unknown', prompt_style='unknown', phase_name='warmup', timestep_offset=0,
    survival_eval_episodes=10, survival_max_steps=None,
):
    history = []
    for round_idx in range(num_rounds):
        rollout = collect_rollout(env, cfg, policy, value_net, reward_model=None, rollout_steps=rollout_steps)
        advantages, returns = compute_gae(rollout['rewards'], rollout['values'], rollout['dones'])
        ppo_update(
            policy, value_net,
            rollout['observations'], rollout['actions'], rollout['logprobs'],
            advantages, returns,
        )

        timestep_end = timestep_offset + (round_idx + 1) * rollout_steps
        avg_env_reward = float(np.mean(rollout['env_rewards'])) if len(rollout['env_rewards']) else 0.0

        row = {
            'env_name': env_name,
            'prompt_style': prompt_style,
            'phase': phase_name,
            'round': round_idx + 1,
            'global_round': round_idx + 1,
            'timesteps_this_round': int(rollout_steps),
            'timestep_end': int(timestep_end),
            'avg_env_reward': avg_env_reward,
            'avg_reward_model_score': float(np.mean(rollout['rewards'])) if len(rollout['rewards']) else 0.0,
            'segments_collected': len(rollout['segments']),
            'new_comparisons': 0,
            'unclear_responses': 0,
            'buffer_size': 0,
            'rm_final_loss': None,
        }
        add_survival_metrics_to_row(
            row,
            env_name=env_name,
            env=env,
            policy=policy,
            survival_eval_episodes=survival_eval_episodes,
            survival_max_steps=survival_max_steps,
        )

        history.append(row)
        print(f'Baseline round {round_idx + 1}/{num_rounds} complete')

    return history


def training_loop_with_prompt_style(
    env, cfg, env_name, policy, value_net, reward_model, buffer, prompt_style, llm_query_fn,
    num_rounds=5, rollout_steps=ROLLOUT_STEPS, phase_name='rlhf', timestep_offset=0,
    global_round_offset=0, survival_eval_episodes=10, survival_max_steps=None,
):
    history = []
    for round_idx in range(num_rounds):
        rollout = collect_rollout(env, cfg, policy, value_net, reward_model=reward_model, rollout_steps=rollout_steps)
        segments = rollout['segments']

        new_comparisons = 0
        unclear = 0

        for _ in range(COMPARISONS_PER_ROUND):
            if len(segments) < 2:
                break
            idx_a, idx_b = np.random.choice(len(segments), size=2, replace=False)
            seg_a, seg_b = segments[idx_a], segments[idx_b]
            preference, _, _ = get_llm_preference_with_style(
                seg_a, seg_b, env_name=env_name, style_name=prompt_style, llm_query_fn=llm_query_fn
            )
            if preference in ['A', 'B']:
                buffer.add(seg_a, seg_b, preference)
                new_comparisons += 1
            else:
                unclear += 1

        rm_final = None
        if len(buffer) >= 5:
            rm_losses = buffer.train_reward_model(reward_model, num_epochs=RM_TRAIN_EPOCHS, lr=LR_REWARD)
            rm_final = rm_losses[-1] if rm_losses else None

        advantages, returns = compute_gae(rollout['rewards'], rollout['values'], rollout['dones'])
        ppo_update(
            policy, value_net,
            rollout['observations'], rollout['actions'], rollout['logprobs'],
            advantages, returns,
        )

        timestep_end = timestep_offset + (round_idx + 1) * rollout_steps
        row = {
            'env_name': env_name,
            'prompt_style': prompt_style,
            'phase': phase_name,
            'round': round_idx + 1,
            'global_round': global_round_offset + round_idx + 1,
            'timesteps_this_round': int(rollout_steps),
            'timestep_end': int(timestep_end),
            'avg_reward_model_score': float(np.mean(rollout['rewards'])) if len(rollout['rewards']) else 0.0,
            'avg_env_reward': float(np.mean(rollout['env_rewards'])) if len(rollout['env_rewards']) else 0.0,
            'segments_collected': len(segments),
            'new_comparisons': new_comparisons,
            'unclear_responses': unclear,
            'buffer_size': len(buffer),
        }
        if rm_final is not None:
            row['rm_final_loss'] = rm_final

        add_survival_metrics_to_row(
            row,
            env_name=env_name,
            env=env,
            policy=policy,
            survival_eval_episodes=survival_eval_episodes,
            survival_max_steps=survival_max_steps,
        )

        history.append(row)
        print(f'Prompt style: {prompt_style} | round {round_idx + 1}/{num_rounds} complete')

    return history


def evaluate_policy_with_env_reward(env, cfg, policy, value_net, rollout_steps=ROLLOUT_STEPS, survival_eval_episodes=10, survival_max_steps=None, training_env_name=None):
    rollout = collect_rollout(env, cfg, policy, value_net, reward_model=None, rollout_steps=rollout_steps)
    segs = rollout['segments']

    # Keep summary focused on non-survival aggregate eval metrics.
    metrics = {
        'env_avg_reward': float(np.mean(rollout['env_rewards'])) if len(rollout['env_rewards']) else 0.0,
        'env_total_reward': float(np.sum(rollout['env_rewards'])) if len(rollout['env_rewards']) else 0.0,
        'avg_time_upright': float(np.mean([s['time_upright'] for s in segs])) if segs else 0.0,
        'segments_collected': len(segs),
        'fell_segments': int(np.sum([1 for s in segs if s['fell']])) if segs else 0,
        'avg_forward_metric': float(np.mean([s['avg_forward_metric'] for s in segs])) if segs else 0.0,
        'forward_metric_name': segs[0]['forward_metric_name'] if segs else 'n/a',
    }

    return metrics


def compare_prompt_styles_on_env_reward_with_warmup(
    env_name, styles, llm_query_fn, warmup_rounds=5, rlhf_rounds=5,
    training_rollout_steps=ROLLOUT_STEPS, eval_rollout_steps=ROLLOUT_STEPS,
    seed=123, return_training_curve=False, return_json_payload=False,
    title=None, description=None,
    include_summary_in_payload=True,
    details_extra=None,
    survival_eval_episodes_per_round=10,
    survival_max_steps_per_round=None,
    include_ppo_only_baseline=True,
    ppo_only_label='ppo_only_policy',
):
    cfg = ENV_CONFIGS[env_name]
    results = []
    training_rows = []

    for i, style in enumerate(styles):
        run_seed = seed + i
        set_global_seed(run_seed)

        env, _ = make_env(env_name, seed=run_seed)
        policy = PolicyNetwork(cfg['obs_dim'], cfg['action_dim'])
        value_net = ValueNetwork(cfg['obs_dim'])
        reward_model = RewardModel(cfg['obs_dim'], cfg['action_dim'])
        buffer = ComparisonBuffer()

        print(f'\nEnv: {env_name} | Style: {style} | Phase 1: PPO warm-up ({warmup_rounds} rounds)')
        warmup_history = training_loop_no_llm(
            env=env,
            cfg=cfg,
            policy=policy,
            value_net=value_net,
            num_rounds=warmup_rounds,
            rollout_steps=training_rollout_steps,
            env_name=env_name,
            prompt_style=style,
            phase_name='warmup',
            timestep_offset=0,
            survival_eval_episodes=survival_eval_episodes_per_round,
            survival_max_steps=survival_max_steps_per_round,
        )

        print(f'Env: {env_name} | Style: {style} | Phase 2: RLHF ({rlhf_rounds} rounds)')
        rlhf_history = training_loop_with_prompt_style(
            env=env,
            cfg=cfg,
            env_name=env_name,
            policy=policy,
            value_net=value_net,
            reward_model=reward_model,
            buffer=buffer,
            prompt_style=style,
            llm_query_fn=llm_query_fn,
            num_rounds=rlhf_rounds,
            rollout_steps=training_rollout_steps,
            phase_name='rlhf',
            timestep_offset=warmup_rounds * training_rollout_steps,
            global_round_offset=warmup_rounds,
            survival_eval_episodes=survival_eval_episodes_per_round,
            survival_max_steps=survival_max_steps_per_round,
        )

        training_rows.extend(warmup_history)
        training_rows.extend(rlhf_history)

        eval_metrics = evaluate_policy_with_env_reward(
            env=env, cfg=cfg, policy=policy, value_net=value_net, rollout_steps=eval_rollout_steps,
            training_env_name=env_name
        )
        eval_metrics['env_name'] = env_name
        eval_metrics['prompt_style'] = style
        eval_metrics['warmup_rounds'] = warmup_rounds
        eval_metrics['rlhf_rounds'] = rlhf_rounds
        eval_metrics['buffer_size'] = len(buffer)
        results.append(eval_metrics)

        env.close()

    if include_ppo_only_baseline:
        ppo_seed = seed + 10000
        set_global_seed(ppo_seed)

        env, _ = make_env(env_name, seed=ppo_seed)
        policy = PolicyNetwork(cfg['obs_dim'], cfg['action_dim'])
        value_net = ValueNetwork(cfg['obs_dim'])

        total_rounds = warmup_rounds + rlhf_rounds
        print(f'\nEnv: {env_name} | PPO-only baseline ({total_rounds} rounds)')
        ppo_only_history = training_loop_no_llm(
            env=env,
            cfg=cfg,
            policy=policy,
            value_net=value_net,
            num_rounds=total_rounds,
            rollout_steps=training_rollout_steps,
            env_name=env_name,
            prompt_style=ppo_only_label,
            phase_name='ppo_only',
            timestep_offset=0,
            survival_eval_episodes=survival_eval_episodes_per_round,
            survival_max_steps=survival_max_steps_per_round,
        )
        training_rows.extend(ppo_only_history)

        ppo_eval_metrics = evaluate_policy_with_env_reward(
            env=env,
            cfg=cfg,
            policy=policy,
            value_net=value_net,
            rollout_steps=eval_rollout_steps,
            training_env_name=env_name,
        )
        ppo_eval_metrics['env_name'] = env_name
        ppo_eval_metrics['prompt_style'] = ppo_only_label
        ppo_eval_metrics['warmup_rounds'] = total_rounds
        ppo_eval_metrics['rlhf_rounds'] = 0
        ppo_eval_metrics['buffer_size'] = 0
        results.append(ppo_eval_metrics)

        env.close()

    summary_df = pd.DataFrame(results).sort_values('env_total_reward', ascending=False).reset_index(drop=True)
    curve_df = pd.DataFrame(training_rows)

    if not curve_df.empty:
        curve_df = curve_df.sort_values(['env_name', 'prompt_style', 'timestep_end']).reset_index(drop=True)

    if return_json_payload:
        payload_title = title if title is not None else f'{env_name.title()} Reward-vs-Timestep Curve'
        payload_description = description if description is not None else (
            f'Per-round training curve data for {env_name} using PPO warm-up followed by RLHF.'
        )

        payload = {
            'title': payload_title,
            'description': payload_description,
            'config': {
                'env_name': env_name,
                'warmup_rounds': warmup_rounds,
                'rlhf_rounds': rlhf_rounds,
                'training_rollout_steps': training_rollout_steps,
                'eval_rollout_steps': eval_rollout_steps,
                'seed': seed,
                'styles': list(styles),
                'survival_eval_episodes_per_round': survival_eval_episodes_per_round,
                'include_ppo_only_baseline': include_ppo_only_baseline,
                'ppo_only_label': ppo_only_label,
            },
            'details': {
                'num_styles': int(len(styles)),
                'total_training_rounds_per_style': int(warmup_rounds + rlhf_rounds),
                'total_training_timesteps_per_style': int((warmup_rounds + rlhf_rounds) * training_rollout_steps),
                'includes_warmup_phase': True,
                'includes_rlhf_phase': True,
                'includes_ppo_only_baseline': bool(include_ppo_only_baseline),
            },
            'data_points': curve_df.to_dict(orient='records'),
        }

        if details_extra is not None and isinstance(details_extra, dict):
            payload['details'].update(details_extra)

        if include_summary_in_payload:
            payload['summary'] = summary_df.to_dict(orient='records')

        return payload

    if return_training_curve:
        return summary_df, curve_df
    return summary_df

## Run Setup

In [ ]:
# LLM setup (same query pattern as RLLLMJOEL.ipynb)

from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import os
    HF_TOKEN = os.getenv('HF_TOKEN')

MODEL_ID = "google/gemma-2-2b-it"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

print("Loading model ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    device_map="auto",
    dtype=torch.bfloat16,
 )

print(f"Model loaded on: {model.device}")
print(f"Model dtype:     {model.dtype}")

def query_llm(prompt, max_new_tokens=100):
    messages = [{"role": "user", "content": prompt}]

    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    input_length = input_ids["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    return response

In [ ]:
# Hopper: Longer training run (~45-60 min target) with PPO warm-up + RLHF

import json

ENV_NAME = 'hopper'
styles_to_test = list(PROMPT_STYLES_BY_ENV['hopper'].keys())

# Requested settings
LONG_WARMUP_ROUNDS = 8
LONG_RLHF_ROUNDS = 8
LONG_TRAIN_ROLLOUT_STEPS = 3072
LONG_EVAL_ROLLOUT_STEPS = 3072

# Keep PPO epochs fixed as requested
PPO_EPOCHS = 4

# Optional increase for reward-model fitting during RLHF
RM_TRAIN_EPOCHS = 12

hopper_long_run_payload = compare_prompt_styles_on_env_reward_with_warmup(
    env_name=ENV_NAME,
    styles=styles_to_test,
    llm_query_fn=query_llm,
    warmup_rounds=LONG_WARMUP_ROUNDS,
    rlhf_rounds=LONG_RLHF_ROUNDS,
    training_rollout_steps=LONG_TRAIN_ROLLOUT_STEPS,
    eval_rollout_steps=LONG_EVAL_ROLLOUT_STEPS,
    seed=123,
    return_json_payload=True,
    title='Hopper Long-Run Reward-vs-Timestep Curve',
    description='Per-round training curve data for Hopper long run with PPO warm-up followed by RLHF.',
)

print('\nHopper long-run JSON payload:')
print(json.dumps(hopper_long_run_payload, indent=2))

In [ ]:
# Pendulum: equivalent longer training run (same structure)

import json

ENV_NAME = 'pendulum'
styles_to_test = list(PROMPT_STYLES_BY_ENV['pendulum'].keys())

# Suggested settings for similar runtime target
LONG_WARMUP_ROUNDS = 8
LONG_RLHF_ROUNDS = 8
LONG_TRAIN_ROLLOUT_STEPS = 3072
LONG_EVAL_ROLLOUT_STEPS = 3072

PPO_EPOCHS = 4
RM_TRAIN_EPOCHS = 12

pendulum_long_run_payload = compare_prompt_styles_on_env_reward_with_warmup(
    env_name=ENV_NAME,
    styles=styles_to_test,
    llm_query_fn=query_llm,
    warmup_rounds=LONG_WARMUP_ROUNDS,
    rlhf_rounds=LONG_RLHF_ROUNDS,
    training_rollout_steps=LONG_TRAIN_ROLLOUT_STEPS,
    eval_rollout_steps=LONG_EVAL_ROLLOUT_STEPS,
    seed=123,
    return_json_payload=True,
    title='Pendulum Long-Run Reward-vs-Timestep Curve',
    description='Per-round training curve data for Pendulum long run with PPO warm-up followed by RLHF.',
)

print('\nPendulum long-run JSON payload:')
print(json.dumps(pendulum_long_run_payload, indent=2))

In [ ]:
# Ant: equivalent longer training run (same structure)

import json

ENV_NAME = 'ant'
styles_to_test = list(PROMPT_STYLES_BY_ENV['ant'].keys())

LONG_WARMUP_ROUNDS = 8
LONG_RLHF_ROUNDS = 8
LONG_TRAIN_ROLLOUT_STEPS = 3072
LONG_EVAL_ROLLOUT_STEPS = 3072

PPO_EPOCHS = 4
RM_TRAIN_EPOCHS = 12

ant_long_run_payload = compare_prompt_styles_on_env_reward_with_warmup(
    env_name=ENV_NAME,
    styles=styles_to_test,
    llm_query_fn=query_llm,
    warmup_rounds=LONG_WARMUP_ROUNDS,
    rlhf_rounds=LONG_RLHF_ROUNDS,
    training_rollout_steps=LONG_TRAIN_ROLLOUT_STEPS,
    eval_rollout_steps=LONG_EVAL_ROLLOUT_STEPS,
    seed=123,
    return_json_payload=True,
    title='Ant Long-Run Reward-vs-Timestep Curve',
    description='Per-round training curve data for Ant long run with PPO warm-up followed by RLHF.',
)

print('\nAnt long-run JSON payload:')
print(json.dumps(ant_long_run_payload, indent=2))

In [ ]:
# Quick-run variant for debugging (both environments, shorter runtime)
import json

QUICK_WARMUP_ROUNDS = 2
QUICK_RLHF_ROUNDS = 2
QUICK_TRAIN_ROLLOUT_STEPS = 1024
QUICK_EVAL_ROLLOUT_STEPS = 1024

quick_hopper_df, quick_hopper_curve_df = compare_prompt_styles_on_env_reward_with_warmup(
    env_name='hopper',
    styles=list(PROMPT_STYLES_BY_ENV['hopper'].keys()),
    llm_query_fn=query_llm,
    warmup_rounds=QUICK_WARMUP_ROUNDS,
    rlhf_rounds=QUICK_RLHF_ROUNDS,
    training_rollout_steps=QUICK_TRAIN_ROLLOUT_STEPS,
    eval_rollout_steps=QUICK_EVAL_ROLLOUT_STEPS,
    seed=202,
    return_training_curve=True,
 )

quick_pendulum_df, quick_pendulum_curve_df = compare_prompt_styles_on_env_reward_with_warmup(
    env_name='pendulum',
    styles=list(PROMPT_STYLES_BY_ENV['pendulum'].keys()),
    llm_query_fn=query_llm,
    warmup_rounds=QUICK_WARMUP_ROUNDS,
    rlhf_rounds=QUICK_RLHF_ROUNDS,
    training_rollout_steps=QUICK_TRAIN_ROLLOUT_STEPS,
    eval_rollout_steps=QUICK_EVAL_ROLLOUT_STEPS,
    seed=303,
    return_training_curve=True,
 )

quick_ant_df, quick_ant_curve_df = compare_prompt_styles_on_env_reward_with_warmup(
    env_name='ant',
    styles=list(PROMPT_STYLES_BY_ENV['ant'].keys()),
    llm_query_fn=query_llm,
    warmup_rounds=QUICK_WARMUP_ROUNDS,
    rlhf_rounds=QUICK_RLHF_ROUNDS,
    training_rollout_steps=QUICK_TRAIN_ROLLOUT_STEPS,
    eval_rollout_steps=QUICK_EVAL_ROLLOUT_STEPS,
    seed=404,
    return_training_curve=True,
)

print('Quick Ant summary results:')
display(quick_ant_df)

print('Quick Ant training curve:')
display(quick_ant_curve_df[['env_name', 'prompt_style', 'phase', 'round', 'global_round', 'timestep_end', 'avg_env_reward', 'avg_reward_model_score']])

print('Quick Hopper summary results:')
display(quick_hopper_df)

print('Quick Hopper training curve (reward vs timestep data):')
display(quick_hopper_curve_df[['env_name', 'prompt_style', 'phase', 'round', 'global_round', 'timestep_end', 'avg_env_reward', 'avg_reward_model_score']])

print('Quick Pendulum summary results:')
display(quick_pendulum_df)

print('Quick Pendulum training curve (reward vs timestep data):')
display(quick_pendulum_curve_df[['env_name', 'prompt_style', 'phase', 'round', 'global_round', 'timestep_end', 'avg_env_reward', 'avg_reward_model_score']])

quick_hopper_payload = compare_prompt_styles_on_env_reward_with_warmup(
    env_name='hopper',
    styles=list(PROMPT_STYLES_BY_ENV['hopper'].keys()),
    llm_query_fn=query_llm,
    warmup_rounds=QUICK_WARMUP_ROUNDS,
    rlhf_rounds=QUICK_RLHF_ROUNDS,
    training_rollout_steps=QUICK_TRAIN_ROLLOUT_STEPS,
    eval_rollout_steps=QUICK_EVAL_ROLLOUT_STEPS,
    seed=202,
    return_json_payload=True,
    title='Quick Hopper Reward-vs-Timestep Curve',
    description='Per-round training curve data for Hopper quick run with PPO warm-up followed by RLHF.',
)

quick_pendulum_payload = compare_prompt_styles_on_env_reward_with_warmup(
    env_name='pendulum',
    styles=list(PROMPT_STYLES_BY_ENV['pendulum'].keys()),
    llm_query_fn=query_llm,
    warmup_rounds=QUICK_WARMUP_ROUNDS,
    rlhf_rounds=QUICK_RLHF_ROUNDS,
    training_rollout_steps=QUICK_TRAIN_ROLLOUT_STEPS,
    eval_rollout_steps=QUICK_EVAL_ROLLOUT_STEPS,
    seed=303,
    return_json_payload=True,
    title='Quick Pendulum Reward-vs-Timestep Curve',
    description='Per-round training curve data for Pendulum quick run with PPO warm-up followed by RLHF.',
)

quick_ant_payload = compare_prompt_styles_on_env_reward_with_warmup(
    env_name='ant',
    styles=list(PROMPT_STYLES_BY_ENV['ant'].keys()),
    llm_query_fn=query_llm,
    warmup_rounds=QUICK_WARMUP_ROUNDS,
    rlhf_rounds=QUICK_RLHF_ROUNDS,
    training_rollout_steps=QUICK_TRAIN_ROLLOUT_STEPS,
    eval_rollout_steps=QUICK_EVAL_ROLLOUT_STEPS,
    seed=404,
    return_json_payload=True,
    title='Quick Ant Reward-vs-Timestep Curve',
    description='Per-round training curve data for Ant quick run with PPO warm-up followed by RLHF.',
)

print('\nQuick Ant JSON payload:')
print(json.dumps(quick_ant_payload, indent=2))

print('\nQuick Hopper JSON payload:')
print(json.dumps(quick_hopper_payload, indent=2))

print('\nQuick Pendulum JSON payload:')
print(json.dumps(quick_pendulum_payload, indent=2))